# LSTM Next-Word Prediction

**Goal:** given the start of a sentence (e.g. `"I am living in"`), predict the most likely next word.

**Pipeline:**
`Text → Tokenizer → n-gram sequences → Pad → (X, y) split → One-hot y → Embedding → LSTM → Dropout → LSTM → Dense (softmax) → Next word`

> This uses a one-sentence toy corpus so every step is easy to follow. The idea is the same for a big dataset.

## 1. Imports

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer            # words -> integers
from tensorflow.keras.preprocessing.sequence import pad_sequences    # make all sequences the same length

## 2. Training text
Our entire dataset is one sentence. Real projects would load thousands of sentences here.

In [ ]:
text = "I am living in India"

## 3. Tokenize: turn words into numbers
Neural networks work with numbers, not words. The `Tokenizer` gives each unique word an integer ID
(most frequent first; ties go in order of appearance). It also lowercases everything.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])        # learn the vocabulary from our text
print(tokenizer.word_index)           # {'i': 1, 'am': 2, 'living': 3, 'in': 4, 'india': 5}

# +1 because ID 0 is reserved for padding, so it is not a real word
total_words = len(tokenizer.word_index) + 1
print(total_words)                    # 6

## 4. Build n-gram sequences
One sentence gives us several training examples. For each position, we take the words **up to that point**,
and the last word in each piece is what the model must learn to predict:

| Sequence | Meaning |
|----------|---------|
| `[1, 2]` | "I am" |
| `[1, 2, 3]` | "I am living" |
| `[1, 2, 3, 4]` | "I am living in" |
| `[1, 2, 3, 4, 5]` | "I am living in India" |

In [ ]:
input_sequence = []

for line in text.split('\n'):                              # go through the text line by line
    token_list = tokenizer.texts_to_sequences([line])[0]    # "I am living in India" -> [1, 2, 3, 4, 5]
    for end in range(1, len(token_list)):                   # end = how far into the sentence we go
        n_gram_sequence = token_list[:end + 1]              # first (end + 1) words
        input_sequence.append(n_gram_sequence)

input_sequence

## 5. Pad the sequences
The model needs inputs of equal length, so we add zeros at the **front** (`padding='pre'`) until every
sequence is as long as the longest one. Padding in front keeps the real words right next to the
prediction point, which suits an LSTM.

```
[1, 2]          ->  [0, 0, 0, 1, 2]
[1, 2, 3]       ->  [0, 0, 1, 2, 3]
[1, 2, 3, 4]    ->  [0, 1, 2, 3, 4]
[1, 2, 3, 4, 5] ->  [1, 2, 3, 4, 5]
```

In [ ]:
max_sequence_len = max([len(x) for x in input_sequence])    # 5
input_sequence = np.array(pad_sequences(input_sequence, maxlen=max_sequence_len, padding='pre'))
input_sequence

## 6. Split into inputs (X) and target (y)
- **predictors (X):** every word except the last, which is what the model sees
- **target (y):** the last word, which is what the model must predict

```
X = [0, 0, 0, 1]   ->  y = 2    ("I" -> "am")
X = [0, 0, 1, 2]   ->  y = 3    ("I am" -> "living")
X = [0, 1, 2, 3]   ->  y = 4    ("I am living" -> "in")
X = [1, 2, 3, 4]   ->  y = 5    ("I am living in" -> "india")
```

In [ ]:
predictors, target = input_sequence[:, :-1], input_sequence[:, -1]
print(predictors)
print(target)

## 7. One-hot encode the target
We are doing **classification**: which of the `total_words` words comes next? So each target ID becomes a
vector of length 6 with a single 1 (e.g. word 2 -> `[0, 0, 1, 0, 0, 0]`).

In [ ]:
target = tf.keras.utils.to_categorical(target, num_classes=total_words)
target

## 8. Build the model

| Layer | What it does | Output shape |
|-------|--------------|--------------|
| `Embedding(6, 100)` | Turns each word ID into a learned 100-number vector | (batch, 4, 100) |
| `LSTM(150, return_sequences=True)` | Reads the words in order; passes its output at **every** step to the next layer | (batch, 4, 150) |
| `Dropout(0.2)` | Randomly switches off 20% of units during training to reduce overfitting | (batch, 4, 150) |
| `LSTM(100)` | Reads that sequence and returns only the **final** summary | (batch, 100) |
| `Dense(6, softmax)` | Gives a probability for each word in the vocabulary (they sum to 1) | (batch, 6) |

**Early stopping:** stops training when the loss stops improving for 3 epochs in a row and goes back to the best weights.
We watch `loss` here because this toy dataset has no validation data. With a bigger dataset, use
`validation_split=0.2` in `fit()` and switch to `monitor='val_loss'`.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

early_stopping = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)

model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_len - 1))   # input is 4 words long (5 minus the target)
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

# categorical_crossentropy matches our one-hot targets; adam is a good default optimizer
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## 9. Train

In [ ]:
history = model.fit(predictors, target, epochs=50, callbacks=[early_stopping])

## 10. Predict the next word
Steps inside the function:
1. Convert the seed text to IDs
2. Keep only the last `max_sequence_len - 1` words if it is too long
3. Pad it to the length the model expects
4. Get the probability of every word, then pick the top few

In [ ]:
def predict_next_word(model, tokenizer, seed_text, max_sequence_len, num_predictors=3):
    # 1. words -> IDs, e.g. "I am living in" -> [1, 2, 3, 4]
    token_list = tokenizer.texts_to_sequences([seed_text])[0]

    # 2. the model only accepts max_sequence_len - 1 words, so keep the most recent ones
    if len(token_list) > max_sequence_len - 1:
        token_list = token_list[-(max_sequence_len - 1):]

    # 3. pad to the model's input length (same 'pre' padding as in training)
    padded_sequence = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')

    # 4. probabilities for every word in the vocabulary (index 0 is the padding slot)
    predictions = model.predict(padded_sequence)[0]

    # argsort() sorts ascending, so take the last N and reverse to get the highest first
    top_indices = predictions.argsort()[-num_predictors:][::-1]

    # reverse lookup: ID -> word
    word_map = {v: k for k, v in tokenizer.word_index.items()}

    predicted_words = []
    predicted_prob = []
    for idx in top_indices:
        if idx in word_map:          # skips index 0, which is padding and not a real word
            predicted_words.append(word_map[idx])
            predicted_prob.append(predictions[idx])
    return predicted_words, predicted_prob

In [ ]:
seed_text = "I am living in"
max_sequence_len = model.input_shape[1] + 1    # recover the original length (model input is 1 shorter)
next_words, probabilities = predict_next_word(model, tokenizer, seed_text, max_sequence_len)

In [ ]:
print('seed text:', seed_text)
print('predicted next words:', next_words)
print('probabilities:', probabilities)

## Takeaways
- With only one sentence, the model memorizes it, so expect `india` to rank first. A real corpus is needed for real generalization.
- Next steps: larger dataset, `validation_split` with `monitor='val_loss'`, generate whole sentences by predicting in a loop.